In [1]:
import requests
import pandas as pd
import json
from datetime import datetime
import time

In [2]:
def fetch_all_fda_recals(search_term="product_type:\"food\"", max_total=10000):
    
    base_url = "https://api.fda.gov/food/enforcement.json"
    all_results = []
    skip = 0
    limit = 1000  # Max per call
    total_fetched = 0
    
    while total_fetched < max_total:
        params = {
            "limit": limit,
            "skip": skip,
            "search": search_term
        }
        
        print(f"Fetching batch: skip={skip}, limit={limit}")
        response = requests.get(base_url, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching batch at skip={skip}: {response.status_code} - {response.text}")
            break
        
        data = response.json()
        batch_results = data.get('results', [])
        if not batch_results:
            print("No more data available.")
            break
        
        all_results.extend(batch_results)
        total_fetched += len(batch_results)
        print(f"Fetched {len(batch_results)} records (total: {total_fetched})")
        
        # Next batch
        skip += limit
        
        # Rate limit
        time.sleep(1)
      
        if len(batch_results) < limit:
            break
    
    return all_results

In [3]:
def flatten_recall(recall):
    flat = {
        'recall_number': recall.get('recall_number', ''),
        'event_id': recall.get('openfda', {}).get('event_id', [{}])[0].get('id', '') if recall.get('openfda', {}).get('event_id') else '',
        'product_type': recall.get('product_type', ''),
        'product_description': recall.get('product_description', ''),
        'reason_for_recall': recall.get('reason_for_recall', ''),
        'voluntary_mandated': recall.get('voluntary_mandated', ''),
        'initial_firm_notification': recall.get('initial_firm_notification', ''),
        'product_quantity': recall.get('product_quantity', ''),
        'recall_initiation_date': recall.get('recall_initiation_date', ''),
        'distribution_pattern': recall.get('distribution_pattern', ''),
        'product_code': recall.get('product_code', ''),
        'recalling_firm': recall.get('recalling_firm', ''),
        'recalling_firm_address': recall.get('recalling_firm_address', ''),
        'recall_status': recall.get('recall_status', ''),
        'classification': recall.get('classification', ''),
        'code_info': json.dumps(recall.get('code_info', [])) if recall.get('code_info') else '',
        'state_member_state': recall.get('state_member_state', ''),
        'country': recall.get('country', ''),
        'secondary_recipient': json.dumps(recall.get('secondary_recipient', [])) if recall.get('secondary_recipient') else '',
        'product': json.dumps(recall.get('product', [])) if recall.get('product') else ''
    }
    return flat

In [4]:
if __name__ == "__main__":
    # Fetch almost all data
    recalls = fetch_all_fda_recals(max_total=10000)
    
    if recalls:
        # Flatten
        flat_data = [flatten_recall(recall) for recall in recalls]
        
        # Create DataFrame
        df = pd.DataFrame(flat_data)
        
        # Save to CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"fda_food_recals_all_{timestamp}.csv"
        df.to_csv(filename, index=False)
        
        print(f"\nDownloaded {len(recalls)} total recalls to {filename}")
        print("\nSample rows:")
        print(df.head(3))
        print(f"\nColumns: {df.columns.tolist()}")
        print(f"Unique reasons for recall (top 10): {df['reason_for_recall'].value_counts().head(10).to_dict()}")
    else:
        print("No data fetched. Check API or parameters.")

Fetching batch: skip=0, limit=1000
Fetched 1000 records (total: 1000)
Fetching batch: skip=1000, limit=1000
Fetched 1000 records (total: 2000)
Fetching batch: skip=2000, limit=1000
Fetched 1000 records (total: 3000)
Fetching batch: skip=3000, limit=1000
Fetched 1000 records (total: 4000)
Fetching batch: skip=4000, limit=1000
Fetched 1000 records (total: 5000)
Fetching batch: skip=5000, limit=1000
Fetched 1000 records (total: 6000)
Fetching batch: skip=6000, limit=1000
Fetched 1000 records (total: 7000)
Fetching batch: skip=7000, limit=1000
Fetched 1000 records (total: 8000)
Fetching batch: skip=8000, limit=1000
Fetched 1000 records (total: 9000)
Fetching batch: skip=9000, limit=1000
Fetched 1000 records (total: 10000)

Downloaded 10000 total recalls to fda_food_recals_all_20251201_141029.csv

Sample rows:
  recall_number event_id product_type  \
0   F-0276-2017                  Food   
1   F-0865-2017                  Food   
2   F-0609-2015                  Food   

                  